# Lab 15: Energy Landscapes

## Quadratic forms, curvature, and optimization

This lab is a computational companion to Chapter 15. The main idea is that a symmetric matrix creates an energy landscape

$$
q(x)=x^T A x.
$$

You will use Python to see bowls, saddles, flat valleys, rotated ellipses, gradient descent paths, least-squares landscapes, and high-dimensional quadratic optimization.

This lab is more than quick practice. It is meant to help students connect formulas, pictures, and algorithms.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


## 1. A quadratic form as a function from vectors to numbers

A quadratic form takes a vector and returns a scalar:

$$
q(x)=x^T A x.
$$

The value can be interpreted as energy, cost, or weighted squared length.


In [ ]:
A = np.array([[3, 1],
              [1, 2]], dtype=float)

vectors = [np.array([1, 0]), np.array([0, 1]), np.array([1, 1]), np.array([2, -1])]

for x in vectors:
    q = x.T @ A @ x
    print(f"x = {x},  q(x) = {q:.3f}")


### Student task

Change the matrix $A$. Try one positive definite matrix, one indefinite matrix, and one singular positive semidefinite matrix. Observe how the values of $q(x)$ change.


## 2. Contour plots: seeing equal-energy curves

A contour plot shows points with equal energy. For positive definite matrices, contours are ellipses. For indefinite matrices, the contours reveal saddle geometry.


In [ ]:
def quadratic_grid(A, lim=3, n=300):
    xs = np.linspace(-lim, lim, n)
    ys = np.linspace(-lim, lim, n)
    X, Y = np.meshgrid(xs, ys)
    Z = A[0,0]*X**2 + (A[0,1]+A[1,0])*X*Y + A[1,1]*Y**2
    return X, Y, Z

def plot_contours(A, title, lim=3):
    X, Y, Z = quadratic_grid(A, lim=lim)
    plt.figure(figsize=(6, 6))
    plt.contour(X, Y, Z, levels=25)
    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.gca().set_aspect('equal')
    plt.title(title)
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.show()

plot_contours(np.array([[4,0],[0,1]]), 'Bowl: positive definite')
plot_contours(np.array([[1,0],[0,-1]]), 'Saddle: indefinite')
plot_contours(np.array([[1,1],[1,1]]), 'Flat valley: positive semidefinite')


## 3. Eigenvalues classify the landscape

For a symmetric matrix:

- all eigenvalues positive $\Rightarrow$ positive definite bowl;
- all eigenvalues nonnegative $\Rightarrow$ positive semidefinite bowl or flat valley;
- mixed signs $\Rightarrow$ saddle;
- all eigenvalues negative $\Rightarrow$ upside-down bowl.


In [ ]:
def classify_symmetric(A, tol=1e-10):
    eigvals = np.linalg.eigvalsh(A)
    if np.all(eigvals > tol):
        label = 'positive definite: strict bowl'
    elif np.all(eigvals >= -tol) and np.any(np.abs(eigvals) <= tol):
        label = 'positive semidefinite: bowl with flat direction(s)'
    elif np.all(eigvals < -tol):
        label = 'negative definite: upside-down bowl'
    elif np.any(eigvals > tol) and np.any(eigvals < -tol):
        label = 'indefinite: saddle'
    else:
        label = 'borderline / numerically unclear'
    return eigvals, label

matrices = {
    'A1': np.array([[2,0],[0,5]], dtype=float),
    'A2': np.array([[1,0],[0,-2]], dtype=float),
    'A3': np.array([[1,1],[1,1]], dtype=float),
    'A4': np.array([[3,1],[1,3]], dtype=float),
}

for name, M in matrices.items():
    eigvals, label = classify_symmetric(M)
    print(name)
    print(M)
    print('eigenvalues:', eigvals)
    print(label)
    print()


## 4. Rotated bowls and principal directions

Cross terms such as $2bxy$ rotate the bowl. Eigenvectors show the true axes of the landscape.


In [ ]:
A = np.array([[3, 1.5],
              [1.5, 2]], dtype=float)

eigvals, eigvecs = np.linalg.eigh(A)
print('Eigenvalues:', eigvals)
print('Eigenvectors as columns:')
print(eigvecs)

plot_contours(A, 'Rotated bowl with eigenvector axes')

# Add eigenvectors to contour plot
X, Y, Z = quadratic_grid(A, lim=3)
plt.figure(figsize=(6,6))
plt.contour(X, Y, Z, levels=25)
for i in range(2):
    v = eigvecs[:, i]
    plt.arrow(0, 0, 2*v[0], 2*v[1], head_width=0.08, length_includes_head=True)
    plt.arrow(0, 0, -2*v[0], -2*v[1], head_width=0.08, length_includes_head=True)
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.gca().set_aspect('equal')
plt.title('Eigenvectors are the principal axes')
plt.show()


## 5. Quadratic functions with linear terms

A common optimization problem has the form

$$
f(x)=\frac12 x^T A x-b^T x.
$$

If $A$ is positive definite, the minimizer solves

$$
Ax=b.
$$


In [ ]:
A = np.array([[4, 1], [1, 2]], dtype=float)
b = np.array([1, 2], dtype=float)
x_star = np.linalg.solve(A, b)
print('minimizer x* =', x_star)
print('gradient at x* =', A @ x_star - b)

def f_value(x):
    return 0.5 * x.T @ A @ x - b.T @ x

print('f(x*) =', f_value(x_star))


## 6. Gradient descent on a quadratic bowl

Gradient descent uses the update

$$
x_{k+1}=x_k-\alpha(Ax_k-b).
$$

The step size $\alpha$ matters. The eigenvalues of $A$ control the safe range and speed.


In [ ]:
def gradient_descent(A, b, x0, alpha, steps=50):
    x = x0.astype(float).copy()
    path = [x.copy()]
    values = []
    for k in range(steps):
        values.append(0.5*x.T@A@x - b.T@x)
        grad = A @ x - b
        x = x - alpha * grad
        path.append(x.copy())
    values.append(0.5*x.T@A@x - b.T@x)
    return np.array(path), np.array(values)

A = np.array([[12, 0], [0, 1]], dtype=float)
b = np.array([1, 1], dtype=float)
x0 = np.array([-2, 2], dtype=float)

for alpha in [0.05, 0.12, 0.18]:
    path, values = gradient_descent(A, b, x0, alpha, steps=35)
    print('alpha =', alpha, 'final x =', path[-1], 'final value =', values[-1])


In [ ]:
A = np.array([[12, 0], [0, 1]], dtype=float)
b = np.array([1, 1], dtype=float)
x0 = np.array([-2, 2], dtype=float)
alpha = 0.12
path, values = gradient_descent(A, b, x0, alpha, steps=35)

xs = np.linspace(-2.5, 1.2, 300)
ys = np.linspace(-0.5, 2.5, 300)
X, Y = np.meshgrid(xs, ys)
Z = 0.5*(A[0,0]*X**2 + A[1,1]*Y**2) - b[0]*X - b[1]*Y

plt.figure(figsize=(6,6))
plt.contour(X, Y, Z, levels=30)
plt.plot(path[:,0], path[:,1], marker='o')
plt.gca().set_aspect('equal')
plt.title('Gradient descent path')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.show()

plt.figure(figsize=(6,4))
plt.plot(values, marker='o')
plt.xlabel('iteration')
plt.ylabel('objective value')
plt.title('Energy decreases during gradient descent')
plt.show()


## 7. Condition number: round bowl versus narrow valley

The condition number of a positive definite matrix is

$$
\kappa(A)=\frac{\lambda_{\max}}{\lambda_{\min}}.
$$

A large condition number means the landscape is narrow and gradient descent may zigzag.


In [ ]:
def run_condition_experiment(kappa):
    A = np.array([[kappa, 0], [0, 1]], dtype=float)
    b = np.array([0, 0], dtype=float)
    x0 = np.array([2, 2], dtype=float)
    alpha = 1.8 / kappa
    path, values = gradient_descent(A, b, x0, alpha, steps=60)
    return A, path, values, alpha

for kappa in [2, 10, 50, 200]:
    A, path, values, alpha = run_condition_experiment(kappa)
    print(f'kappa={kappa:3d}, alpha={alpha:.4f}, final norm={np.linalg.norm(path[-1]):.4e}')


## 8. Least squares as an energy landscape

Least squares minimizes

$$
E(w)=\|Xw-y\|^2.
$$

This is a quadratic energy landscape. Its curvature is controlled by $X^T X$.


In [ ]:
rng = np.random.default_rng(7)
n = 40
xdata = np.linspace(0, 5, n)
ydata = 1.5 + 0.8*xdata + rng.normal(0, 0.5, size=n)

X = np.column_stack([np.ones(n), xdata])
w_hat = np.linalg.lstsq(X, ydata, rcond=None)[0]
print('least-squares coefficients:', w_hat)

plt.figure(figsize=(6,4))
plt.scatter(xdata, ydata)
plt.plot(xdata, X @ w_hat)
plt.xlabel('input feature')
plt.ylabel('output')
plt.title('Least-squares line')
plt.show()

H = 2 * X.T @ X
print('Hessian eigenvalues:', np.linalg.eigvalsh(H))
print('condition number:', np.linalg.cond(H))


## 9. Visualizing the least-squares landscape

The parameters are $w_0$ and $w_1$. Each point in parameter space represents a line.


In [ ]:
w0_grid = np.linspace(w_hat[0]-2, w_hat[0]+2, 150)
w1_grid = np.linspace(w_hat[1]-1, w_hat[1]+1, 150)
W0, W1 = np.meshgrid(w0_grid, w1_grid)
Loss = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        w = np.array([W0[i,j], W1[i,j]])
        Loss[i,j] = np.mean((X@w - ydata)**2)

plt.figure(figsize=(6,5))
plt.contour(W0, W1, Loss, levels=30)
plt.scatter([w_hat[0]], [w_hat[1]], marker='x', s=80)
plt.xlabel('$w_0$')
plt.ylabel('$w_1$')
plt.title('Least-squares loss landscape in parameter space')
plt.show()


## 10. High-dimensional quadratic optimization

In high dimensions, we cannot draw the landscape directly. But eigenvalues still describe its shape.

The next experiment constructs a positive definite matrix with a controlled spectrum and observes gradient descent.


In [ ]:
def make_spd_with_spectrum(eigs, seed=0):
    rng = np.random.default_rng(seed)
    M = rng.normal(size=(len(eigs), len(eigs)))
    Q, _ = np.linalg.qr(M)
    return Q @ np.diag(eigs) @ Q.T

n = 80
eigs_good = np.linspace(1, 5, n)
eigs_bad = np.geomspace(1, 500, n)

for name, eigs in [('well conditioned', eigs_good), ('ill conditioned', eigs_bad)]:
    A = make_spd_with_spectrum(eigs, seed=3)
    b = np.ones(n)
    x0 = np.zeros(n)
    alpha = 1.8 / eigs.max()
    path, values = gradient_descent(A, b, x0, alpha, steps=200)
    print(name)
    print('condition number:', eigs.max()/eigs.min())
    print('final objective:', values[-1])
    print()


In [ ]:
plt.figure(figsize=(6,4))
for name, eigs in [('well conditioned', eigs_good), ('ill conditioned', eigs_bad)]:
    A = make_spd_with_spectrum(eigs, seed=3)
    b = np.ones(n)
    x0 = np.zeros(n)
    alpha = 1.8 / eigs.max()
    path, values = gradient_descent(A, b, x0, alpha, steps=200)
    plt.plot(values - values.min() + 1e-12, label=name)
plt.yscale('log')
plt.xlabel('iteration')
plt.ylabel('objective gap, shifted log scale')
plt.title('Conditioning affects gradient descent speed')
plt.legend()
plt.show()


## 11. Reflection questions

1. Why do positive definite matrices create unique minimizers?
2. Why does an indefinite matrix produce a saddle?
3. How does a cross term $2bxy$ rotate the landscape?
4. Why is $X^T X$ positive semidefinite in least squares?
5. Why do high condition numbers slow down gradient descent?

## 12. Mini-project

Choose one of the following:

1. Create a family of $2\times2$ symmetric matrices and animate how the contour plot changes as one entry changes.
2. Compare gradient descent with three different step sizes on the same quadratic function.
3. Generate a synthetic linear regression dataset with badly scaled features and show how standardization improves the condition number of $X^T X$.
4. Build a high-dimensional positive definite matrix with chosen eigenvalues and study how the eigenvalue distribution affects convergence.
